# RQ1 Virality Modeling

This notebook reproduces the RQ1 ablation study from `notebook_04_modeling.ipynb`.
It trains XGBoost models to predict whether a Reddit post becomes viral using:

- text-only features
- structural-only features
- combined features

It also saves the ablation summary, plots, and SHAP-based explanations.

### Required files

- `outputs/features.parquet`
- `outputs/feature_groups.json`

Run this in Google Colab, mount your Drive or upload the project files and set the working directory to the project root.


In [ ]:
# Install dependencies if needed
!pip install -q xgboost shap pandas scikit-learn matplotlib pyarrow


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE    = '/content/drive/MyDrive/Social_Media_Mining_Project'

Mounted at /content/drive


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
    r2_score,
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier, XGBRegressor

In [ ]:
DATA_DIR = Path(BASE) / "outputs"
FEATURE_FILE = DATA_DIR / "features.parquet"
FEATURE_GROUP_FILE = DATA_DIR / "feature_groups.json"
OUT_DIR = DATA_DIR
RANDOM_SEED = 6
TEST_SIZE = 0.20
VIRAL_LABEL = "is_viral"
REGRESSION_TARGET = "engagement_score"
STRATIFY_COLUMN = "subreddit"

In [ ]:
def load_data():
    if not FEATURE_FILE.exists():
        raise FileNotFoundError(f"Missing features file: {FEATURE_FILE}")
    if not FEATURE_GROUP_FILE.exists():
        raise FileNotFoundError(f"Missing feature groups file: {FEATURE_GROUP_FILE}")

    df = pd.read_parquet(FEATURE_FILE)
    with FEATURE_GROUP_FILE.open("r", encoding="utf-8") as f:
        feature_groups = json.load(f)

    for col in [REGRESSION_TARGET, VIRAL_LABEL, STRATIFY_COLUMN]:
        if col not in df.columns:
            raise KeyError(f"Required column not found in {FEATURE_FILE}: {col}")

    return df, feature_groups

In [ ]:
def train_and_evaluate(df, feature_groups):
    indices = np.arange(len(df))
    train_idx, test_idx = train_test_split(
        indices,
        test_size=TEST_SIZE,
        random_state=RANDOM_SEED,
        stratify=df[STRATIFY_COLUMN].values,
    )

    y_reg = df[REGRESSION_TARGET].values
    y_cls = df[VIRAL_LABEL].values

    configs = {
        "text_only": feature_groups["text_lexical"] + feature_groups["text_semantic"],
        "structural_only": feature_groups["structural"],
        "combined": feature_groups["text_lexical"]
        + feature_groups["text_semantic"]
        + feature_groups["structural"],
    }

    xgb_params = dict(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=0,
    )

    results = {}
    for config_name, feat_cols in configs.items():
        print(f"--- Training {config_name} ({len(feat_cols)} features) ---")

        X = df[feat_cols].fillna(0).values
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr_reg, y_te_reg = y_reg[train_idx], y_reg[test_idx]
        y_tr_cls, y_te_cls = y_cls[train_idx], y_cls[test_idx]

        reg = XGBRegressor(**xgb_params)
        reg.fit(X_tr, y_tr_reg)
        y_pred_reg = reg.predict(X_te)
        rmse = np.sqrt(mean_squared_error(y_te_reg, y_pred_reg))
        mae = mean_absolute_error(y_te_reg, y_pred_reg)
        r2 = r2_score(y_te_reg, y_pred_reg)
        print(f"  Regression  RMSE={rmse:.4f}  MAE={mae:.4f}  R2={r2:.4f}")

        clf = XGBClassifier(
            **xgb_params, eval_metric="logloss", use_label_encoder=False
        )
        clf.fit(X_tr, y_tr_cls)
        y_pred_cls = clf.predict(X_te)
        y_prob_cls = clf.predict_proba(X_te)[:, 1]
        acc = accuracy_score(y_te_cls, y_pred_cls)
        f1 = f1_score(y_te_cls, y_pred_cls)
        auc_score = roc_auc_score(y_te_cls, y_prob_cls)
        print(f"  Classification  Acc={acc:.4f}  F1={f1:.4f}  AUC={auc_score:.4f}")

        results[config_name] = {
            "feat_cols": feat_cols,
            "reg_model": reg,
            "clf_model": clf,
            "rmse": rmse,
            "mae": mae,
            "r2": r2,
            "acc": acc,
            "f1": f1,
            "auc": auc_score,
        }

    return results, train_idx, test_idx

In [ ]:
def save_ablation_summary(results):
    summary = pd.DataFrame(
        {
            config: {
                "n_features": len(value["feat_cols"]),
                "RMSE": round(value["rmse"], 4),
                "MAE": round(value["mae"], 4),
                "R2": round(value["r2"], 4),
                "Acc": round(value["acc"], 4),
                "F1": round(value["f1"], 4),
                "AUC": round(value["auc"], 4),
            }
            for config, value in results.items()
        }
    ).T

    out_path = OUT_DIR / "rq1_ablation_results.csv"
    summary.to_csv(out_path)
    print(f"Saved ablation summary -> {out_path}")
    return summary


def plot_ablation(summary):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    metrics = ["AUC", "F1", "R2"]
    colors = ["#4C72B0", "#DD8452", "#55A868"]

    for ax, metric, color in zip(axes, metrics, colors):
        vals = summary[metric].astype(float)
        ax.bar(vals.index, vals.values, color=color, edgecolor="white", lw=0.5)
        ax.set_title(metric)
        ax.set_ylabel(metric)
        ax.set_ylim(0, 1)
        for i, v in enumerate(vals.values):
            ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=10)
        ax.set_xticklabels(vals.index, rotation=15)

    plt.suptitle("RQ1 Ablation: text vs structural vs combined", fontsize=14, y=1.02)
    plt.tight_layout()
    path = OUT_DIR / "rq1_ablation_comparison.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Saved ablation plot -> {path}")
    plt.close(fig)

In [ ]:
def explain_with_shap(df, results, test_idx):
    try:
        import shap
    except ImportError:
        print("SHAP is not installed. Skipping explanation step.")
        return

    combined_result = results["combined"]
    feat_cols = combined_result["feat_cols"]
    X_test = df[feat_cols].fillna(0).values[test_idx]
    model = combined_result["reg_model"]

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)

    shap_abs = np.abs(shap_values).mean(axis=0)
    shap_df = pd.DataFrame(
        {
            "feature": feat_cols,
            "mean_abs_shap": shap_abs,
        }
    ).sort_values("mean_abs_shap", ascending=False)
    out_path = OUT_DIR / "rq1_shap_top20.csv"
    shap_df.head(20).to_csv(out_path, index=False)
    print(f"Saved SHAP top 20 -> {out_path}")

    fig, ax = plt.subplots(figsize=(10, 7))
    top20 = shap_df.head(20)[::-1]
    ax.barh(top20["feature"], top20["mean_abs_shap"], color="#4C72B0")
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_title("RQ1 SHAP top 20 features")
    plt.tight_layout()
    plot_path = OUT_DIR / "rq1_shap_top20.png"
    fig.savefig(plot_path, dpi=150, bbox_inches="tight")
    print(f"Saved SHAP bar plot -> {plot_path}")
    plt.close(fig)

    shap.summary_plot(
        shap_values, pd.DataFrame(X_test, columns=feat_cols), max_display=20, show=False
    )
    beeswarm_path = OUT_DIR / "rq1_shap_beeswarm.png"
    plt.savefig(beeswarm_path, dpi=150, bbox_inches="tight")
    print(f"Saved SHAP beeswarm -> {beeswarm_path}")
    plt.close()

In [ ]:
def classification_report_for_combined(df, results, test_idx):
    combined_clf = results["combined"]["clf_model"]
    feat_cols = results["combined"]["feat_cols"]
    X_test = df[feat_cols].fillna(0).values[test_idx]
    y_true = df[VIRAL_LABEL].values[test_idx]
    y_pred = combined_clf.predict(X_test)
    report = classification_report(y_true, y_pred, target_names=["Non-viral", "Viral"])
    print("\n=== Combined classification report ===")
    print(report)

    with (OUT_DIR / "rq1_classification_report.txt").open("w", encoding="utf-8") as f:
        f.write(report)
    print(f"Saved classification report -> {OUT_DIR / 'rq1_classification_report.txt'}")

In [ ]:
df, feature_groups = load_data()
results, train_idx, test_idx = train_and_evaluate(df, feature_groups)
summary = save_ablation_summary(results)
print("\nAblation summary:", summary)
plot_ablation(summary)
explain_with_shap(df, results, test_idx)
classification_report_for_combined(df, results, test_idx)
print("\nRQ1 modeling complete.")

--- Training text_only (61 features) ---
  Regression  RMSE=0.0907  MAE=0.0702  R2=0.4473
  Classification  Acc=0.8210  F1=0.2760  AUC=0.8024
--- Training structural_only (16 features) ---
  Regression  RMSE=0.0455  MAE=0.0338  R2=0.8607
  Classification  Acc=0.8991  F1=0.7392  AUC=0.9482
--- Training combined (77 features) ---
  Regression  RMSE=0.0441  MAE=0.0328  R2=0.8695
  Classification  Acc=0.9071  F1=0.7529  AUC=0.9526
Saved ablation summary -> /content/drive/MyDrive/Social_Media_Mining_Project/outputs/rq1_ablation_results.csv

Ablation summary:                  n_features    RMSE     MAE      R2     Acc      F1     AUC
text_only              61.0  0.0907  0.0702  0.4473  0.8210  0.2760  0.8024
structural_only        16.0  0.0455  0.0338  0.8607  0.8991  0.7392  0.9482
combined               77.0  0.0441  0.0328  0.8695  0.9071  0.7529  0.9526


/tmp/ipykernel_4698/2076766089.py:36: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(vals.index, rotation=15)
/tmp/ipykernel_4698/2076766089.py:36: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(vals.index, rotation=15)
/tmp/ipykernel_4698/2076766089.py:36: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(vals.index, rotation=15)


Saved ablation plot -> /content/drive/MyDrive/Social_Media_Mining_Project/outputs/rq1_ablation_comparison.png
Saved SHAP top 20 -> /content/drive/MyDrive/Social_Media_Mining_Project/outputs/rq1_shap_top20.csv
Saved SHAP bar plot -> /content/drive/MyDrive/Social_Media_Mining_Project/outputs/rq1_shap_top20.png
Saved SHAP beeswarm -> /content/drive/MyDrive/Social_Media_Mining_Project/outputs/rq1_shap_beeswarm.png

=== Combined classification report ===
              precision    recall  f1-score   support

   Non-viral       0.93      0.95      0.94      7905
       Viral       0.79      0.72      0.75      1943

    accuracy                           0.91      9848
   macro avg       0.86      0.84      0.85      9848
weighted avg       0.90      0.91      0.91      9848

Saved classification report -> /content/drive/MyDrive/Social_Media_Mining_Project/outputs/rq1_classification_report.txt

RQ1 modeling complete.
